# Baku Bus Ridership Forecasting

Explore daily passenger counts by route, check data quality, and compare a Random Forest forecast with simple historical baselines. Run this notebook from top to bottom. The included dataset ends on 2026-08-31; its next-day forecast is historical, not a current prediction.

In [39]:
import pandas as pd
import numpy as np


In [40]:
data=pd.read_csv('bakubus_passenger-2026.csv')

In [41]:
data.head(10)

,Tarix (il-ay-gün),Marşrutun adı,Gündəlik gediş sayı
0,2026-01-01,5,12030
1,2026-01-01,125,11937
2,2026-01-01,62,11498
3,2026-01-01,88,10766
4,2026-01-01,7A,8012
5,2026-01-01,46,7939
6,2026-01-01,120,7738
7,2026-01-01,14,7616
8,2026-01-01,21,7551
9,2026-01-01,3,7496


In [42]:
data.shape

(13801, 3)

In [43]:
data = data.rename(columns={
    "Tarix (il-ay-gün)": "date",
    "Marşrutun adı": "route",
    "Gündəlik gediş sayı": "passengers"
})

In [44]:
data.head(10)

,date,route,passengers
0,2026-01-01,5,12030
1,2026-01-01,125,11937
2,2026-01-01,62,11498
3,2026-01-01,88,10766
4,2026-01-01,7A,8012
5,2026-01-01,46,7939
6,2026-01-01,120,7738
7,2026-01-01,14,7616
8,2026-01-01,21,7551
9,2026-01-01,3,7496


In [45]:
data = data.replace(r"^\s*$", np.nan, regex=True)
data["date"] = pd.to_datetime(data["date"], errors="coerce")
data["passengers"] = pd.to_numeric(data["passengers"], errors="coerce")
data["route"] = data["route"].astype("string").str.strip()

In [46]:
print(data.isna().sum())

date          0
route         0
passengers    0
dtype: int64


In [47]:
duplicate_count = data.duplicated(["date", "route"]).sum()
print("Duplicate route/date rows:", duplicate_count)

Duplicate route/date rows: 0


In [48]:
all_dates = pd.date_range(data["date"].min(), data["date"].max(), freq="D")
all_routes = data["route"].dropna().unique()

expected = pd.MultiIndex.from_product(
    [all_dates, all_routes], names=["date", "route"]
)
observed = pd.MultiIndex.from_frame(data[["date", "route"]].drop_duplicates())
missing_route_dates = expected.difference(observed)

print("Missing route/date rows:", len(missing_route_dates))
print("Examples:", list(missing_route_dates[:10]))

Missing route/date rows: 1994
Examples: [(Timestamp('2026-01-01 00:00:00'), '149E'), (Timestamp('2026-01-01 00:00:00'), '150E'), (Timestamp('2026-01-01 00:00:00'), '195E'), (Timestamp('2026-01-01 00:00:00'), '206'), (Timestamp('2026-01-01 00:00:00'), 'M2'), (Timestamp('2026-01-01 00:00:00'), 'M3'), (Timestamp('2026-01-01 00:00:00'), 'M4'), (Timestamp('2026-01-01 00:00:00'), 'M5'), (Timestamp('2026-01-01 00:00:00'), 'M6'), (Timestamp('2026-01-02 00:00:00'), '149E')]


In [49]:
# Sort the data and make sure each date/route appears once
data = data.sort_values(["date", "route"]).copy()

if data.duplicated(["date", "route"]).any():
    raise ValueError("Resolve duplicate date/route rows before modeling.")

data = data.copy()

# Add the same route's count from exactly 7 and 14 calendar days earlier
for lag in [7, 14]:
    lagged = data[["date", "route", "passengers"]].copy()
    lagged["date"] = lagged["date"] + pd.Timedelta(days=lag)
    lagged = lagged.rename(columns={"passengers": f"lag_{lag}"})

    data = data.merge(
        lagged,
        on=["date", "route"],
        how="left",
        validate="one_to_one"
    )

data["weekday"] = data["date"].dt.dayofweek  # Monday=0, Sunday=6
data["is_weekend"] = data["weekday"].isin([5, 6]).astype(int)

# Keep rows where both historical counts are available
data = data.dropna(subset=["lag_7", "lag_14"]).copy()

print("Rows available for modeling:", len(data))

Rows available for modeling: 12859


In [50]:
cutoff = data["date"].max() - pd.Timedelta(days=27)

train = data[data["date"] < cutoff]
test = data[data["date"] >= cutoff]

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Test period:", test["date"].min(), "to", test["date"].max())

Training rows: 11215
Testing rows: 1644
Test period: 2026-08-04 00:00:00 to 2026-08-31 00:00:00


In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

features = ["route", "weekday", "is_weekend", "lag_7", "lag_14"]
categorical_features = ["route"]
numeric_features = ["weekday", "is_weekend", "lag_7", "lag_14"]

preprocessor = ColumnTransformer(
    transformers=[
        ("route", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numbers", "passthrough", numeric_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(train[features], train["passengers"])

model_predictions = model.predict(test[features])
baseline_predictions = test["lag_7"].to_numpy()
actual = test["passengers"].to_numpy()

print("Model MAE:", mean_absolute_error(actual, model_predictions))
print("7-day baseline MAE:", mean_absolute_error(actual, baseline_predictions))

average_lag_predictions = test[["lag_7", "lag_14"]].mean(axis=1)
print("Average-lag baseline MAE:", mean_absolute_error(actual, average_lag_predictions))

Model MAE: 766.5192488211093
7-day baseline MAE: 674.9476885644768
Average-lag baseline MAE: 714.2329683698297


In [52]:
results = test[["route", "passengers", "lag_7", "lag_14"]].copy()
results["model_prediction"] = model_predictions
results["average_lag_prediction"] = average_lag_predictions

results["model_error"] = (results["passengers"] - results["model_prediction"]).abs()
results["baseline_error"] = (results["passengers"] - results["lag_7"]).abs()
results["average_lag_error"] = (
    results["passengers"] - results["average_lag_prediction"]
).abs()

by_route = results.groupby("route")[["model_error", "baseline_error", "average_lag_error"]].mean()
by_route["model_improvement_vs_7_day"] = (
    by_route["baseline_error"] - by_route["model_error"]
)

print("Routes where the model improves most over the 7-day baseline:")
display(by_route.sort_values("model_improvement_vs_7_day", ascending=False).head(10))

print("Routes where the model performs worst versus the 7-day baseline:")
display(by_route.sort_values("model_improvement_vs_7_day").head(10))

Routes where the model improves most over the 7-day baseline:


,model_error,baseline_error,average_lag_error,model_improvement_vs_7_day
route,,,,
M2,390.002967,728.250000,1956.500000,338.247033
33,938.946477,1071.250000,1159.125000,132.303523
M6,345.710074,477.333333,474.833333,131.623259
20,645.800526,763.142857,603.553571,117.342331
24,528.699977,643.178571,573.928571,114.478595
88A,717.507071,817.857143,802.500000,100.350072
125,862.565778,940.321429,721.339286,77.755651
21,573.358069,649.571429,627.142857,76.213360
205,450.448579,521.321429,376.642857,70.872850


Routes where the model performs worst versus the 7-day baseline:


,model_error,baseline_error,average_lag_error,model_improvement_vs_7_day
route,,,,
7A,5238.269134,3526.857143,4766.750000,-1711.411991
195E,1064.395004,86.380952,93.476190,-978.014052
14,4521.910898,3861.821429,4517.750000,-660.089470
M3,838.395854,298.750000,2367.750000,-539.645854
13,1359.652913,997.785714,1201.464286,-361.867199
49,1203.648047,863.642857,800.946429,-340.005190
88,2066.419277,1737.857143,2102.910714,-328.562134
3,1095.800367,777.142857,845.428571,-318.657509
62,910.312157,629.857143,663.642857,-280.455014


In [53]:
tomorrow = data["date"].max() + pd.Timedelta(days=1)

future = pd.DataFrame({
    "date": tomorrow,
    "route": data["route"].dropna().unique()
})

for lag in [7, 14]:
    lagged = data[["date", "route", "passengers"]].copy()
    lagged["date"] = lagged["date"] + pd.Timedelta(days=lag)
    lagged = lagged.rename(columns={"passengers": f"lag_{lag}"})

    future = future.merge(
        lagged,
        on=["date", "route"],
        how="left",
        validate="one_to_one"
    )

future["weekday"] = future["date"].dt.dayofweek
future["is_weekend"] = future["weekday"].isin([5, 6]).astype(int)

ready = future.dropna(subset=["lag_7", "lag_14"]).copy()
ready["predicted_passengers"] = np.maximum(
    0,
    model.predict(ready[features])
)

ready[["date", "route", "predicted_passengers"]].head(10)

,date,route,predicted_passengers
0,2026-09-01,1,15274.205336
1,2026-09-01,10,10837.617719
2,2026-09-01,108A,4817.403184
3,2026-09-01,11,14947.454952
4,2026-09-01,120,11531.696419
5,2026-09-01,120E,3318.606689
6,2026-09-01,121E,4523.357731
7,2026-09-01,125,14588.051263
8,2026-09-01,13,18449.712741
9,2026-09-01,130,3122.259618
